# 第二章 LangChain核心组件实操

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # 加载环境变量
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")

### 2.1.2 实操案例1：统一接口调用不同模型

#### （1）调用OpenAI的ChatModel

In [ ]:
from langchain_openai import ChatOpenAI

# 1. 初始化对话模型
# 不管是哪个厂商的ChatModel，初始化参数都类似（model、temperature等）
chat_model = ChatOpenAI(
    model="deepseek-chat",  # 选择对话模型
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.3,        # 随机性：0-1，越小越严谨，越大越有创造力
    max_tokens=200          # 最大生成 tokens 数，避免生成过长内容
)

In [37]:
# 2. 构造对话消息
# ChatModel需要接收的是“消息列表”，每个消息有角色（user/assistant/system）和内容
messages = [
    # system消息：给助手设定身份和行为准则，会影响后续所有回复
    {"role": "system", "content": "你是一个耐心的AI学习助手，回复简洁易懂，适合高校学生理解。"},
    # user消息：用户的问题
    {"role": "user", "content": "请用3句话解释什么是LangChain？"}
]

In [38]:
# 3. 调用模型生成结果
# 统一调用方法：invoke()，传入消息列表
result = chat_model.invoke(messages)

In [33]:
# 4. 输出结果
# 结果是一个ChatMessage对象，content属性是回复内容
print("ChatModel回复：")
print(result.content)

ChatModel回复：
LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架。它提供了一系列工具和模块，帮助开发者将语言模型与外部数据源、记忆机制和业务逻辑连接起来。通过 LangChain，可以更方便地创建如智能问答、对话系统或自动化代理等复杂应用。


#### （3）快速切换到Hugging Face模型

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 1. 加载Hugging Face的模型和tokenizer
model_name = r"C:\Users\Username\Desktop\iii\models\Qwen\Qwen3-0___6B"  # 模型名
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
# 2. 构建pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.3
)

In [ ]:
# 3. 初始化LangChain的LLM接口（统一接口）
hf_llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# 4. 调用模型（和之前的LLM调用方式完全一样）
prompt = "请用3句话解释什么是LangChain？"
result = hf_llm.invoke(prompt)

In [ ]:
print("Hugging Face模型回复：")
print(result)

## 2.2 提示词模板（PromptTemplate）：让提示更规范、可复用

### 2.2.1 提示词模板基础用法：标准化提示与动态参数

In [39]:
from langchain_core.prompts import PromptTemplate

# 1. 定义提示词模板
# input_variables：动态参数列表（这里是user_role和subject）
# template：提示词模板字符串，用{参数名}表示动态参数
prompt_template = PromptTemplate(
    input_variables=["user_role", "subject"],
    template="请给{user_role}写一段50字左右的{subject}学习建议，语言简洁实用，分2个小要点。"
)

In [40]:
# 2. 格式化模板（传入具体参数，生成完整提示词）
# 给“高校学生”生成“LangChain”学习建议
formatted_prompt = prompt_template.format(
    user_role="高校学生",
    subject="LangChain"
)
print("格式化后的提示词：")
print(formatted_prompt)

格式化后的提示词：
请给高校学生写一段50字左右的LangChain学习建议，语言简洁实用，分2个小要点。


In [41]:
# 3. 调用模型生成结果
result = chat_model.invoke([{"role": "user", "content": formatted_prompt}])

print("\n生成的学习建议：")
print(result.content)


生成的学习建议：
1. 先掌握LangChain核心模块（如Chains、Agents、Memory），理解其工作流程。  
2. 动手实践：用真实项目（如问答系统）巩固知识，善用官方文档和社区资源。


### 2.2.2 提示词模板进阶用法：少样本提示模板

In [42]:
# 1. 定义示例（少样本的核心：给模型看的参考案例）
examples = [
    {
        "subject": "Python编程",
        "method": "核心目标：掌握基础语法和常用库；学习步骤：1. 学习变量、函数等基础语法 2. 实操小项目（如计算器） 3. 学习Pandas、Matplotlib库；注意事项：多动手实操，遇到错误及时调试。"
    },
    {
        "subject": "机器学习",
        "method": "核心目标：理解基础算法原理和应用场景；学习步骤：1. 复习数学基础（线性代数、概率） 2. 学习经典算法（线性回归、决策树） 3. 用Scikit-learn实操；注意事项：先理解原理，再动手实现，避免死记硬背。"
    }
]

In [43]:
# 2. 定义示例模板（告诉框架如何解析示例）
example_template = """
学科：{subject}
学习方法：{method}
"""

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# 3. 定义最终的提示词模板（包含示例和用户需求）
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,                # 传入示例
    example_prompt=PromptTemplate(
        input_variables=["subject", "method"],
        template=example_template
    ),
    example_separator="\n",
    prefix="少样本提示词：",
    suffix="参考以上样本，回答：\n学科：{new_subject}\n学习方法：",  # 最终给用户的提示（在示例之后）
    input_variables=["new_subject"]   # 动态参数：用户要查询的新学科
)

In [ ]:
# 4. 格式化模板（传入新学科：LangChain
formatted_prompt = few_shot_prompt.format(new_subject="LangChain")
print(formatted_prompt)

In [46]:
# 5. 调用模型生成结果
result = chat_model.invoke([{"role": "user", "content": formatted_prompt}])

print("生成的LangChain学习方法 (AI回复)：")
print(result.content)

生成的LangChain学习方法 (AI回复)：
学科：LangChain  
学习方法：核心目标：掌握LangChain框架的核心组件与典型应用；学习步骤：1. 理解基本概念（如链、代理、记忆、文档加载器） 2. 实操基础示例（如构建问答系统、调用大模型） 3. 集成外部工具（如向量数据库、API工具）开发完整应用；注意事项：结合实际场景练习，注重理解组件间的协作机制，及时查阅官方文档和社区案例。


### 2.2.3 工程化实践：少样本提示模板的高效管理

In [ ]:
import json

# 2. 工程化示例管理：从JSON文件加载示例（避免硬编码，便于维护）
with open("learning_method_examples.json", "r", encoding="utf-8") as f:
    examples = json.load(f)  # 从JSON中直接提取示例数据列表
# 示例文件格式参考（learning_method_examples.json）：
# [
#   {"subject": "Python编程（入门）", "difficulty": "easy", "method": "核心目标：掌握基础语法；学习步骤：1.变量与数据类型 2.条件语句；注意事项：边学边练"},
#   {"subject": "Python编程（进阶）", "difficulty": "hard", "method": "核心目标：掌握面向对象与库开发；学习步骤：1.类与对象 2.模块开发；注意事项：参与开源项目"},
#   {"subject": "机器学习（入门）", "difficulty": "easy", "method": "核心目标：理解基础概念；学习步骤：1.数据预处理 2.简单模型；注意事项：用Excel辅助理解"},
#   {"subject": "机器学习（进阶）", "difficulty": "hard", "method": "核心目标：掌握模型优化；学习步骤：1.特征工程 2.超参数调优；注意事项：研读论文复现实验"}
# ]

In [ ]:
from langchain_core.example_selectors import BaseExampleSelector
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import PromptTemplate
from typing import Dict, List


# 3. ExampleSelector：按长度筛选示例
# example_selector = LengthBasedExampleSelector(
#     examples=examples,
#     example_prompt=PromptTemplate(
#         input_variables=["subject", "difficulty", "method"],
#         template="学科：{subject}\n难度：{difficulty}\n学习方法：{method}\n"
#     ),
#     max_length=150,  # 控制示例总长度，避免提示词过长
#     get_text_length=lambda x: len(x)  # 长度计算函数
# )

# 3. 自定义ExampleSelector：按难度筛选示例（输入含difficulty参数）
class DifficultyExampleSelector(BaseExampleSelector):
    """根据用户输入的 difficulty 字段筛选样本"""
    def __init__(self, examples: List[Dict[str, str]]):
        self.examples = examples

    def add_example(self, example: Dict[str, str]) -> None:
        self.examples.append(example)

    def select_examples(self, input_variables: Dict[str, str]) -> List[Dict]:
        # 获取用户输入的难度等级，如果没有提供则默认为 'easy'
        target_difficulty = input_variables.get("difficulty", "easy")

        # 过滤出匹配难度的所有示例
        return [ex for ex in self.examples if ex.get("difficulty") == target_difficulty]


example_selector = DifficultyExampleSelector(examples=examples)

In [49]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# 4. 构建工程化少样本模板
few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,  # 替换固定examples为动态选择器
    example_prompt=PromptTemplate(
        input_variables=["subject", "difficulty", "method"],
        template="学科：{subject}\n难度：{difficulty}\n学习方法：{method}\n"
    ),
    example_separator="\n",
    prefix="少样本提示：",
    suffix="参考以上示例，回答：\n学科：{new_subject}\n难度：{difficulty}\n学习方法：",
    input_variables=["new_subject", "difficulty"]  # 新增难度参数
)

In [51]:
# 5. 动态生成不同难度的提示词
# 场景1：生成入门级LangChain学习方法
formatted_prompt_easy = few_shot_prompt.format(
    new_subject="LangChain",
    difficulty="easy"
)

print(formatted_prompt_easy)
result_easy = chat_model.invoke([{"role": "user", "content": formatted_prompt_easy}])
print("\n入门级学习方法 (AI回复)：")
print(result_easy.content)

少样本提示：
学科：Python编程（入门）
难度：easy
学习方法：核心目标：掌握基础语法；学习步骤：1.变量与数据类型 2.条件语句；注意事项：边学边练

学科：机器学习（入门）
难度：easy
学习方法：核心目标：理解基础概念；学习步骤：1.数据预处理 2.简单模型；注意事项：用Excel辅助理解

参考以上示例，回答：
学科：LangChain
难度：easy
学习方法：

入门级学习方法 (AI回复)：
学科：LangChain  
难度：easy  
学习方法：核心目标：掌握基本组件与链式调用；学习步骤：1.安装与环境配置 2.使用PromptTemplate和LLM调用 3.构建简单链（如LLMChain）；注意事项：结合实际小项目（如问答机器人）边学边练


In [52]:
# 场景2：生成进阶级LangChain学习方法
formatted_prompt_hard = few_shot_prompt.format(
    new_subject="LangChain",
    difficulty="hard"
)

print(formatted_prompt_hard)
result_hard = chat_model.invoke([{"role": "user", "content": formatted_prompt_hard}])
print("\n进阶级学习方法 (AI回复)：")
print(result_hard.content)

少样本提示：
学科：Python编程（进阶）
难度：hard
学习方法：核心目标：掌握面向对象与库开发；学习步骤：1.类与对象 2.模块开发；注意事项：参与开源项目

学科：机器学习（进阶）
难度：hard
学习方法：核心目标：掌握模型优化；学习步骤：1.特征工程 2.超参数调优；注意事项：研读论文复现实验

参考以上示例，回答：
学科：LangChain
难度：hard
学习方法：

进阶级学习方法 (AI回复)：
学科：LangChain  
难度：hard  
学习方法：核心目标：掌握大语言模型应用链式编排与智能体开发；学习步骤：1.理解LangChain核心组件（如Chains、Agents、Memory、Retrievers）2.构建端到端复杂应用（如RAG系统、多跳推理、工具调用集成）；注意事项：深入阅读官方文档与源码，复现前沿LangChain相关论文或GitHub高星项目，积极参与社区讨论与贡献。


## 2.3 输出解析：让输出更可控

#### 2.3.2.1 案例1：StrOutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# 1. 创建 StrOutputParser
# 核心作用：将 LLM 返回的 AIMessage 对象，统一转为纯字符串（str）
parser = StrOutputParser()

In [ ]:
# 2. 链式调用：模型 → 字符串解析
chain = chat_model | parser
result = chain.invoke("请简要介绍 LangChain 输出解析层的作用")

print("StrOutputParser 解析后的字符串：")
print(result)
print("\n解析结果类型：", type(result))  # str

StrOutputParser 解析后的字符串：
LangChain 的输出解析层（Output Parser）主要用于将大语言模型（LLM）生成的原始文本输出转换为结构化、可编程使用的数据格式。

其核心作用包括：

1. **格式转换**：将 LLM 返回的非结构化文本（如自然语言回答）解析为结构化数据，例如 JSON、字典、列表、自定义对象等。
2. **提高可靠性**：通过定义明确的输出格式（如使用 Pydantic 模型或固定模板），减少模型输出的不确定性，提升下游应用的稳定性。
3. **简化集成**：使开发者能直接在代码中使用解析后的数据，而无需手动处理字符串或正则表达式提取信息。
4. **支持链式调用**：在 LangChain 的 Chain 或 Agent 中，输出解析器可作为中间组件，将一个步骤的输出自动转换为下一个步骤所需的输入格式。

常见的输出解析器包括：
- `StrOutputParser`：仅返回字符串。
- `JsonOutputParser`：解析为 JSON 对象。
- `PydanticOutputParser`：基于 Pydantic 模型进行结构化解析。
- `RegexParser`：使用正则表达式提取特定字段。

总之，输出解析层是连接 LLM 非结构化输出与应用程序结构化需求的关键桥梁。

解析结果类型： <class 'langchain_core.messages.base.TextAccessor'>


#### 2.3.2.2 案例2：JsonOutputParser

In [4]:
from langchain_core.output_parsers import JsonOutputParser

# 1. 创建 JSON 解析器（无需额外配置，默认引导模型输出 JSON）
parser = JsonOutputParser()

In [6]:
from langchain_core.prompts import PromptTemplate

# 2. 构建提示模板（无需手动嵌入格式指令，解析器自动关联）
prompt = PromptTemplate(
    template="请介绍1个LangChain开发工具，输出工具名和核心功能。{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

In [ ]:
# 3. 链式调用（LangChain ≥1.0.0 推荐方式，自动完成提示+调用+解析）
chain = prompt | chat_model | parser
result = chain.invoke({})  # 无输入参数，传入空字典

print("解析后的JSON（Python字典）：")
print(result)
print("获取单个字段：", result["tool_name"])  # 可直接用于业务逻辑

#### 2.3.2.3 案例3：PydanticOutputParser

In [42]:
from pydantic import BaseModel, Field

# 1. 定义 Pydantic 数据模型
class ToolInfo(BaseModel):
    tool_name: str = Field(description="开发工具的名称，如 LangSmith")
    function: str = Field(description="工具的核心功能，30字以内")
    difficulty: str = Field(description="学习难度，仅可选：简单 / 中等 / 复杂")

In [43]:
from langchain_core.output_parsers import PydanticOutputParser

# 2. 创建解析器
parser = PydanticOutputParser(pydantic_object=ToolInfo)

In [ ]:
# 3. Prompt + Chain
prompt = PromptTemplate(
    template="{user_input}，严格按照要求输出。\n{format_instructions}",
    input_variables=["user_input"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

chain = prompt | chat_model | parser
result = chain.invoke({"user_input": "请介绍1个 Python 开发工具"})

In [45]:
print("解析后的结构化数据（Pydantic 模型对象）：")
print(result)

print("字段校验 difficulty：", result.difficulty)

print("转化为字典：", result.model_dump())

解析后的结构化数据（Pydantic 模型对象）：
tool_name='PyCharm' function='集成开发环境，支持代码调试与测试' difficulty='中等'
字段校验 difficulty： 中等
转化为字典： {'tool_name': 'PyCharm', 'function': '集成开发环境，支持代码调试与测试', 'difficulty': '中等'}


### 2.3.3 BaseOutputParser 核心抽象接口

In [33]:
from langchain_core.output_parsers import BaseOutputParser

# 自定义解析器
class CustomToolParser(BaseOutputParser):
    def parse(self, text: str) -> dict:
        """将模型输出按 '工具名@核心功能@学习难度' 解析为字典"""
        text = text.strip().replace("\n", "").replace(" ", "")
        parts = text.split("@")
        if len(parts) != 3:
            raise ValueError(f"输出格式错误！需满足「工具名@核心功能@学习难度」，当前输出：{text}")
        return {
            "tool_name": parts[0].strip(),
            "function": parts[1].strip(),
            "difficulty": parts[2].strip()
        }

    def get_format_instructions(self) -> str:
        """生成提示词，引导模型按自定义格式输出"""
        return "请严格按照「工具名@核心功能@学习难度」格式输出，不添加多余内容。示例：LangSmith@全链路调试监控@中等"

In [34]:
# 使用解析器
parser = CustomToolParser()
prompt = PromptTemplate(
    template="请介绍1个LangChain开发工具。{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
chain = prompt | chat_model | parser
result = chain.invoke({})

print("自定义解析器解析结果：")
print(result)
print("解析结果类型：", type(result))

自定义解析器解析结果：
{'tool_name': 'LangGraph', 'function': '构建状态化、多智能体工作流', 'difficulty': '中等'}
解析结果类型： <class 'dict'>
